# LegaLoom-Env: GRPO Training

Single-phase training on `task_hard` (inoperative-PAN scenarios) with `num_generations=8`.

**Works on both Colab T4 (~50 min) and HF JupyterLab Space A10G (~3 hrs).**

- Colab: uses Unsloth for fast training
- HF Space: uses vanilla HF + PEFT + bitsandbytes (Unsloth has a known dtype bug on A10G)

Outputs: `reward_curves.png`, `before_after.png`, `training_log.json`, `training_scores.json`

In [ ]:
# Cell 1: Detect platform + install
import os

ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
print(f'Platform: {"Colab" if ON_COLAB else "HF Space / Local"}')

if ON_COLAB:
    !pip install unsloth 'trl>=0.12.0' openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml datasets matplotlib --quiet
else:
    !pip install -q 'transformers>=4.46,<4.50' 'trl>=0.12.0,<0.16' 'peft>=0.13' \
        bitsandbytes accelerate openenv-core==0.2.3 fastapi uvicorn pydantic httpx \
        openai pyyaml datasets matplotlib

print('✓ Installed')

In [ ]:
# Cell 2: Clone repo into writable location
import sys, os, subprocess

if ON_COLAB:
    WORK_DIR = '/content/legaloom_env'
else:
    WORK_DIR = '/data/legaloom-env' if os.path.exists('/data') and os.access('/data', os.W_OK) else os.path.expanduser('~/legaloom-env')

if not os.path.exists(WORK_DIR):
    subprocess.run(
        ['git', 'clone', 'https://huggingface.co/spaces/aarav0202/legaloom-env', WORK_DIR],
        check=True,
    )

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f'✓ Working dir: {os.getcwd()}')
print(f'✓ Files: {sorted(os.listdir("."))[:8]} ...')

In [ ]:
# Cell 3: Load model — Unsloth on Colab, vanilla HF+PEFT on HF Space
import torch

if ON_COLAB:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen2.5-3B-Instruct',
        max_seq_length=2048, dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
    )
    MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map='auto', torch_dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0, bias='none', task_type='CAUSAL_LM',
    ))

print(f'✓ Model loaded. dtype={next(model.parameters()).dtype}')

In [ ]:
# Cell 4: Baseline measurement — 10 episodes per task
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

if ON_COLAB:
    from unsloth import FastLanguageModel as FLM
    FLM.for_inference(model)
else:
    model.eval()

print('Measuring baseline (untrained), 10 episodes per task...')
baseline_scores = {}
baseline_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    baseline_scores[task_id] = sum(scores) / len(scores)
    baseline_per_episode[task_id] = scores
    std = (sum((s - baseline_scores[task_id])**2 for s in scores) / len(scores))**0.5
    print(f'  {task_id}: mean={baseline_scores[task_id]:.3f}  std={std:.3f}')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')

In [ ]:
# Cell 5: Single-phase GRPO on task_hard (40 steps, num_generations=8)
import os
from train_grpo import build_training_dataset, episode_reward_fn
from trl import GRPOConfig, GRPOTrainer

if ON_COLAB:
    from unsloth import FastLanguageModel as FLM, is_bfloat16_supported
    FLM.for_training(model)
    bf16 = is_bfloat16_supported()
    fp16 = not bf16
    gc_kwargs = {}
    batch_size = 1
else:
    model.train()
    model.enable_input_require_grads()
    bf16 = True
    fp16 = False
    gc_kwargs = {'gradient_checkpointing_kwargs': {'use_reentrant': False}}
    batch_size = 8  # HF Space: batch must be >= num_generations

log_history = []
dataset = build_training_dataset(task_ids=['task_hard'], examples_per_task=120)

def _make_reward_fn(tid):
    def fn(prompts, completions, **kwargs):
        clean_kwargs = {k: v for k, v in kwargs.items() if k != 'task_id'}
        return episode_reward_fn(prompts, completions, task_id=tid, **clean_kwargs)
    return fn

cfg_dict = dict(
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=4 if ON_COLAB else 1,
    num_generations=8,
    max_prompt_length=1536,
    max_completion_length=512,
    beta=0.01,
    logging_steps=1,
    max_steps=40,
    output_dir='./legaloom_grpo_output/task_hard',
    report_to='none',
    bf16=bf16,
    fp16=fp16,
    save_strategy='no',
    gradient_checkpointing=True,
)
cfg_dict.update(gc_kwargs)
cfg = GRPOConfig(**cfg_dict)

trainer = GRPOTrainer(
    model=model, args=cfg, train_dataset=dataset,
    reward_funcs=[_make_reward_fn('task_hard')],
    processing_class=tokenizer,
)
trainer.train()

for entry in trainer.state.log_history:
    entry['phase'] = 0
    entry['phase_task_id'] = 'task_hard'
log_history.extend(trainer.state.log_history)
print(f'\n✓ Done. {len(log_history)} log entries.')

In [ ]:
# Cell 5.5: DIAGNOSTIC — verify training actually updated the model
import torch

print('=== LoRA weight check (B matrices init to zero — nonzero = learned) ===')
for name, param in model.named_parameters():
    if 'lora_B' in name and 'q_proj' in name:
        norm = param.data.norm().item()
        maxv = param.data.abs().max().item()
        status = '✓ LEARNED' if norm > 0.001 else '✗ NO LEARNING (weights still ~zero)'
        print(f'  {name}: norm={norm:.6f}, max={maxv:.6f} → {status}')
        break

print()
print('=== Quick generation test: trained vs base ===')
test_prompt = 'You are a TDS compliance agent. Read the invoice and determine TDS.'
with torch.no_grad():
    inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
    out_trained = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    text_trained = tokenizer.decode(out_trained[0], skip_special_tokens=True)

try:
    with model.disable_adapter():
        out_base = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
        text_base = tokenizer.decode(out_base[0], skip_special_tokens=True)
    same = text_trained == text_base
    print(f'  Outputs identical: {same}')
    if same:
        print('  ⚠ Model may not have learned — outputs identical to base')
    else:
        print('  ✓ Model produces different output — training had effect')
except Exception as e:
    print(f'  Could not compare: {e}')
    print('  Proceeding — LoRA weight norm is the primary check.')

In [ ]:
# Cell 6: Plot reward curves
import matplotlib.pyplot as plt
import json

with open('training_log.json', 'w') as f:
    json.dump(log_history, f, indent=2, default=str)

rewards = [e['reward'] for e in log_history if 'reward' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = list(range(1, len(rewards) + 1))
ax.plot(x, rewards, 'b-', linewidth=1.5, alpha=0.6, label='Episode reward')
if len(rewards) >= 3:
    w = 3
    ma = [sum(rewards[max(0,j-w+1):j+1])/min(j+1,w) for j in range(len(rewards))]
    ax.plot(x, ma, 'r-', linewidth=2.5, label='3-step moving avg')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xlabel('Training Step'); ax.set_ylabel('Episode Reward')
ax.set_title('GRPO — task_hard (40 steps, num_gen=8)')
ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
if losses:
    ax2.plot(range(1, len(losses)+1), losses, 'g-', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Training Step'); ax2.set_ylabel('Loss')
ax2.set_title('GRPO — Loss'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ reward_curves.png saved')

In [ ]:
# Cell 7: Trained model evaluation — 10 episodes per task
if ON_COLAB:
    FLM.for_inference(model)
else:
    model.eval()

# Verify adapter is active
if hasattr(model, 'active_adapters'):
    print(f'Active adapters: {model.active_adapters}')

print('Measuring trained model, 10 episodes per task...')
trained_scores = {}
trained_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    trained_scores[task_id] = sum(scores) / len(scores)
    trained_per_episode[task_id] = scores
    std = (sum((s - trained_scores[task_id])**2 for s in scores) / len(scores))**0.5
    delta = trained_scores[task_id] - baseline_scores[task_id]
    rel = (delta / baseline_scores[task_id] * 100) if baseline_scores[task_id] > 0 else 0
    print(f'  {task_id}: before={baseline_scores[task_id]:.3f}  after={trained_scores[task_id]:.3f}  std={std:.3f}  Δ={delta:+.3f} ({rel:+.0f}%)')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')
print(f'Trained avg:  {sum(trained_scores.values()) / 4:.3f}')
print(f'Lift:         {(sum(trained_scores.values()) - sum(baseline_scores.values())) / 4:+.3f}')

In [ ]:
# Cell 8: Before/After bar chart + save scores
import matplotlib.pyplot as plt
import json

tasks = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after = [trained_scores[t] for t in tasks]

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(tasks)); w = 0.36
ax.bar([i - w/2 for i in x], before, w, label='Before GRPO (untrained)', color='#e76f51')
ax.bar([i + w/2 for i in x], after, w, label='After GRPO (single-phase task_hard)', color='#2a9d8f')
for i in x:
    ax.text(i - w/2, before[i] + 0.015, f'{before[i]:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, after[i] + 0.015, f'{after[i]:.3f}', ha='center', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('Average Score (10 episodes)')
ax.set_title('LegaLoom-Env: Before vs After GRPO Training')
ax.set_ylim(0, 1); ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()

with open('training_scores.json', 'w') as f:
    json.dump({
        'baseline': baseline_scores,
        'trained': trained_scores,
        'baseline_per_episode': baseline_per_episode,
        'trained_per_episode': trained_per_episode,
        'n_episodes_per_task': 10,
        'training_config': 'single-phase task_hard, 40 steps, num_gen=8',
    }, f, indent=2)
print('✓ before_after.png + training_scores.json saved')

In [ ]:
# Cell 9: Print README table + download artifacts
import os

print('=' * 60)
print('PASTE THIS INTO README.md Results section:')
print('=' * 60)
print()
print('| Task | Baseline | After GRPO | Δ |')
print('|------|---------:|-----------:|------:|')
for tid, lbl in [('task_easy','`task_easy`'),('task_medium','`task_medium`'),
                  ('task_hard','`task_hard`'),('task_expert','`task_expert`')]:
    b, a = baseline_scores[tid], trained_scores[tid]
    d = ((a-b)/b*100) if b > 0 else 0
    bold = '**' if a > b else ''
    print(f'| {lbl} | {b:.3f} | {bold}{a:.3f}{bold} | {d:+.0f}% |')
ab = sum(baseline_scores.values())/4
aa = sum(trained_scores.values())/4
ad = ((aa-ab)/ab*100) if ab > 0 else 0
print(f'| **Average** | **{ab:.3f}** | **{aa:.3f}** | **{ad:+.0f}%** |')
print()
print('=' * 60)
print(f'Artifacts in: {os.getcwd()}')
for fname in ['reward_curves.png', 'before_after.png', 'training_scores.json', 'training_log.json']:
    exists = '✓' if os.path.exists(fname) else '✗'
    print(f'  {exists} {fname}')
print()

# Download on Colab, manual on HF Space
if ON_COLAB:
    from google.colab import files
    for fname in ['reward_curves.png', 'before_after.png', 'training_scores.json', 'training_log.json']:
        try:
            files.download(fname)
            print(f'  Downloaded: {fname}')
        except Exception as e:
            print(f'  Download failed: {fname}: {e}')
else:
    print('HF Space: right-click files in JupyterLab browser → Download')
    print('Then: Settings → Hardware → CPU basic to stop GPU billing.')